# Hanssem Macro BigQuery ETL

운영자용 Colab 노트북입니다.

실행 순서:
1. Google 인증
2. 환경변수 설정
3. BigQuery 테이블/뷰 초기화
4. `pipeline run-bq` 실행
5. ETL 실행 이력 및 source verification 확인

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
!pip install -q google-cloud-bigquery pandas-gbq db-dtypes
!pip install -q -e /content/2026-04-30-api-db-etl[bigquery]

## 운영자 설정값

아래 값만 수정하면 됩니다.

In [ ]:
import os

BQ_PROJECT_ID = 'YOUR_PROJECT_ID'
BQ_DATASET = 'hanssem_macro'
BQ_LOCATION = 'asia-northeast3'

KOSIS_API_KEY = 'YOUR_KOSIS_API_KEY'
RONE_API_KEY = 'YOUR_RONE_API_KEY'
ECOS_API_KEY = ''  # optional
DATA_GO_KR_API_KEY = ''  # optional

os.environ['BQ_PROJECT_ID'] = BQ_PROJECT_ID
os.environ['BQ_DATASET'] = BQ_DATASET
os.environ['BQ_LOCATION'] = BQ_LOCATION
os.environ['BIGQUERY_PROJECT_ID'] = BQ_PROJECT_ID
os.environ['BIGQUERY_DATASET'] = BQ_DATASET
os.environ['BIGQUERY_LOCATION'] = BQ_LOCATION

os.environ['KOSIS_API_KEY'] = KOSIS_API_KEY
os.environ['RONE_API_KEY'] = RONE_API_KEY
os.environ['ECOS_API_KEY'] = ECOS_API_KEY
os.environ['DATA_GO_KR_API_KEY'] = DATA_GO_KR_API_KEY

In [ ]:
from hanssem_macro_dashboard.bq import initialize_bigquery_datamart

initialize_bigquery_datamart()
print(f'Initialized BigQuery datamart: {BQ_PROJECT_ID}.{BQ_DATASET}')

## 운영 실행

아래 셀 하나로 운영 ETL을 실행합니다.

In [ ]:
!python -m hanssem_macro_dashboard.pipeline run-bq

## ETL 실행 이력 확인

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=BQ_PROJECT_ID)
query = f"""
SELECT
  run_id,
  indicator_id,
  collect_status,
  stage_status,
  rows_loaded,
  latest_period,
  started_at,
  finished_at,
  message
FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.etl_run_history`
ORDER BY started_at DESC, indicator_id
LIMIT 50
"""
client.query(query).to_dataframe()

## Source Verification 확인

In [ ]:
query = f"""
SELECT
  indicator_id,
  source_name,
  provider,
  verification_status,
  error_type,
  rows,
  last_verified_at,
  message
FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.source_verification`
ORDER BY indicator_id
"""
client.query(query).to_dataframe()

## Tableau 연결 대상 확인

- `vw_hanssem_macro_hmi`
- `vw_macro_sales_join`
- `source_verification`
- `etl_run_history`

In [ ]:
query = f"SELECT * FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.vw_hanssem_macro_hmi` ORDER BY date DESC LIMIT 12"
client.query(query).to_dataframe()

## 개발용 참고

로컬 SQLite 기반 개발/테스트는 아래 명령을 사용합니다.

- `python -m hanssem_macro_dashboard.pipeline demo`
- `python -m hanssem_macro_dashboard.pipeline run`

운영 기준 실행은 항상 `python -m hanssem_macro_dashboard.pipeline run-bq` 입니다.